# 연관 과목 이유 문장 엔진 실험 v1

연구실 상세 화면의 각 과목 옆에 붙는 **"왜 이 과목인가"** 문장을 만드는 방법을 비교한다.
데이터셋이 아직 완성되지 않았으므로, 지금 데이터로 방법론을 미리 검증하고 **데이터가 갖춰지면 그대로 다시 돌릴 수 있는 엔진 구조**를 만든다.

### 엔진 구조 (방법과 무관하게 공통)
```
(연구실, 과목) 쌍
  → ① 근거 선택: 연구실 주제 문장 × 과목 문장 중 가장 잘 맞는 한 쌍 (재순위 모델)
  → ② 문장 생성: 아래 방법 M1~M5 중 하나
  → ③ 검증: 근거에 없는 내용·과장 표현 검사
```

| 방법 | 설명 | 장점 / 위험 |
|---|---|---|
| M1 | 개념 템플릿 (현재 앱 방식): "A·B 지식을 연결해볼 수 있습니다." | 안전 / 모호함 |
| M2 | 근거 템플릿: 연구실 주제 + 과목 근거 문장을 그대로 보여줌 | 사실 그대로 / 길고 딱딱함 |
| M3 | 키워드 템플릿: 근거 문장에서 핵심어 2개를 뽑아 틀에 넣음 | 짧고 구체적 / 어색할 수 있음 |
| M4 | 로컬 LLM 생성 (근거만 보고 1~2문장) | 자연스러움 / 지어낼 위험 |
| M5 | M4 + 검증: 근거에 없는 단어가 많거나 과장 표현이 있으면 M2로 대체 | 자연스러움 + 안전 |

### 평가 (정답 세트 없이)
1. **자동 지표**: 길이, 근거에 없는 단어 비율, 문장 중복률, 과장 표현
2. **Claude Code 블라인드 채점**: 방법 이름을 숨긴 채 정확성(0/1)·구체성·자연스러움·도움(1~3)

### 준비
- **런타임 → 런타임 유형 변경 → T4 GPU**
- 업로드 (서버 저장소 `data/`): `courses.jsonl`, `lab_profiles.jsonl`, `course_profiles.jsonl`, `concepts.json`
- 업로드 (서버 `experiments/bge_m3/V4/`): `rankings_v4.json` — R2 순위를 그대로 사용

In [ ]:
# 1. GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 선택하고 다시 실행하세요.')

In [ ]:
# 2. 라이브러리 설치
!pip install -q sentence-transformers transformers accelerate bitsandbytes

In [ ]:
# 3. 데이터 업로드
import os
from google.colab import files
need = ['courses.jsonl', 'lab_profiles.jsonl', 'course_profiles.jsonl', 'concepts.json', 'rankings_v4.json']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('업로드할 파일:', missing)
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'아직 없는 파일: {missing}'
print('데이터 준비 완료')

In [ ]:
# 4. 데이터와 평가 대상 쌍
import json, re, time, random, statistics
from collections import defaultdict, Counter
import numpy as np

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

courses = load_jsonl('courses.jsonl')
course_by_id = {c['record_id']: c for c in courses}
profiles = [p for p in load_jsonl('lab_profiles.jsonl') if p.get('status') != 'insufficient']
prof_by_id = {p['lab_id']: p for p in profiles}
course_profiles = {c['record_id']: c for c in load_jsonl('course_profiles.jsonl')}
concepts = {c['id']: c for c in json.load(open('concepts.json', encoding='utf-8'))['concepts']}
rankings = json.load(open('rankings_v4.json', encoding='utf-8'))['R2']

N_LABS, TOP_K, SEED = 12, 5, 7          # 연구실 수 × 상위 과목 수 = 평가 쌍 수
labs = random.Random(SEED).sample([p['lab_id'] for p in profiles], 20)[:N_LABS]   # v2~v4 채점 연구실과 같은 순서
pairs = [(lid, c['record_id']) for lid in labs for c in rankings[lid]['top30'][:TOP_K]]
print(f'평가 쌍 {len(pairs)}개 (연구실 {len(labs)}곳 × 상위 {TOP_K}개)')

def course_name(c):
    return c.get('course_name_kr') or c.get('course_name_en') or c['course_code']

## 1단계: 근거 선택 (모든 방법이 공유)

연구실의 주요 주제 문장들 × 과목 문장들(설명·학습목표를 문장으로 나누고, 강의계획은 줄 단위)을 재순위 모델에 넣어
**가장 잘 맞는 (주제 문장, 과목 문장) 한 쌍**을 고른다. 이 한 쌍이 모든 방법의 근거가 된다.

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
if device == 'cuda':
    reranker.model.half()

SENT_SPLIT = re.compile(r'(?<=[.!?。])\s+|(?<=다\.)\s*|\s*[•■▪\-–]\s+|\s*/\s+')
JUNK = re.compile(r'(see attached|attached file|첨부|tbd|tba|mid-?term|final|exam|quiz|homework|assignment|grading|attendance|presentation|review|holiday|no class|assumed|familiar with|prerequisite|중간|기말|시험|퀴즈|과제|출석|성적|평가|발표|휴강|공휴일|선수과목)', re.I)

def course_sentences(c):
    out = []
    for where, text in [('과목 설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]:
        for s in SENT_SPLIT.split(text):
            s = s.strip()
            if 15 <= len(s) <= 300 and not JUNK.search(s):
                out.append((where, s))
    for line in re.split(r'\n+', str(c.get('course_plan') or '')):
        line = re.sub(r'^[\W\d_]*(week|주차|lecture|chapter)?\s*[\d.]*\s*[:.)-]?\s*', '', line.strip(), flags=re.I)
        if 10 <= len(line) <= 200 and not JUNK.search(line):
            out.append(('강의계획', line))
    return list(dict.fromkeys(out))[:60]

def primary_topics(lid):
    return [t for t in prof_by_id[lid]['topics'] if t.get('weight', 'primary') == 'primary']

t0 = time.time()
batch, owners = [], []
for pi, (lid, rid) in enumerate(pairs):
    sents = course_sentences(course_by_id[rid])
    for ti, t in enumerate(primary_topics(lid)):
        for si, (where, s) in enumerate(sents):
            batch.append((f"{t['label_ko']}: {t['quote']}", s))
            owners.append((pi, ti, si))
scores = np.asarray(reranker.predict(batch, batch_size=64, show_progress_bar=True), dtype=np.float32)
best = {}
for (pi, ti, si), sc in zip(owners, scores):
    if pi not in best or sc > best[pi][0]:
        best[pi] = (float(sc), ti, si)

def shared_concepts(lid, rid):
    core = {t['concept'] for t in primary_topics(lid) if t.get('scope', 'core') == 'core'}
    return [concepts[x['concept']]['ko'] for x in course_profiles.get(rid, {}).get('concepts', []) if x['concept'] in core]

evidence = []
for pi, (lid, rid) in enumerate(pairs):
    c = course_by_id[rid]; sents = course_sentences(c)
    if pi in best:
        sc, ti, si = best[pi]; t = primary_topics(lid)[ti]; where, s = sents[si]
    else:
        sc, t, where, s = 0.0, primary_topics(lid)[0], '과목 설명', (c.get('description') or '')[:200]
    evidence.append({'lab_id': lid, 'record_id': rid, 'lab': prof_by_id[lid]['professors'][0].split(' (')[0],
                     'lab_summary': prof_by_id[lid]['summary_ko'], 'topic': t['label_ko'], 'topic_quote': t['quote'],
                     'course': course_name(c), 'course_en': c.get('course_name_en') or '', 'code': c['course_code'],
                     'where': where, 'sentence': s, 'rr': round(sc, 4), 'concepts': shared_concepts(lid, rid)})
print(f'근거 선택 {len(batch)}쌍 {time.time()-t0:.1f}s')
for e in evidence[:4]:
    print(f"  [{e['lab']}] {e['topic']} ↔ {e['course']} ({e['where']}): {e['sentence'][:80]}")

## 2단계: 방법별 문장 생성

- M1~M3은 규칙 기반이라 바로 만들어진다.
- M4는 로컬 LLM이 근거만 보고 쓴다. 기본 모델은 `Qwen/Qwen2.5-7B-Instruct`(4비트, T4 가능, 상업 이용 가능 라이선스).
  한국어 품질 비교가 필요하면 `LLM_ID`를 바꿔 다시 실행한다 (예: `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` — 연구용 라이선스).

In [ ]:
def short(s, n=60):
    s = re.sub(r'\s+', ' ', s).strip()
    return s if len(s) <= n else s[:n].rsplit(' ', 1)[0] + '…'

# M1: 현재 앱 방식
def m1(e):
    if e['concepts']:
        return ' · '.join(dict.fromkeys(e['concepts'])) + ' 지식을 연결해볼 수 있습니다.'
    return '연구실 주제와 관련된 내용을 다룹니다.'

# M2: 근거 템플릿
def m2(e):
    return f"이 연구실의 '{e['topic']}' 연구와 연결돼요. 이 과목의 {e['where']}에서 \"{short(e['sentence'], 70)}\"를 다뤄요."

# M3: 키워드 템플릿 (근거 문장의 후보 구절 중 연구실 주제와 가장 가까운 2개)
embedder = SentenceTransformer('BAAI/bge-m3', device=device)
if device == 'cuda':
    embedder.half()
STOP = {'the', 'and', 'for', 'with', 'this', 'that', 'these', 'those', 'from', 'into', 'are', 'is', 'be', 'to', 'of', 'in', 'on', 'as', 'by', 'an', 'or', 'its', 'their', 'such', 'using', 'used', 'including', 'based', 'about', 'very', 'also', 'can', 'will', 'course', 'class', 'students', 'student', 'learn', 'understand', 'basic', 'introduction', 'various', 'important', 'role', 'problems', 'arising', 'domains', '학습', '이해', '과목', '본', '강의', '통해', '대한', '위한', '기본', '다양한', '기초', '소개', '방법', '학생', '수업', '내용', '목표'}
def phrases(s):
    words = re.findall(r'[A-Za-z][A-Za-z\-]{2,}|[가-힣]{2,}', s)
    words = [w for w in words if w.lower() not in STOP]
    cand = set(words) | {' '.join(words[i:i+2]) for i in range(len(words) - 1)}
    return [c for c in cand if 2 <= len(c) <= 30]
def m3(e):
    cand = phrases(e['sentence'])
    if not cand:
        return m2(e)
    v = embedder.encode([f"{e['topic']}: {e['topic_quote']}"] + cand, normalize_embeddings=True)
    order = np.argsort(-(v[1:] @ v[0]))
    picked = []
    for i in order:
        if all(cand[i] not in p and p not in cand[i] for p in picked):
            picked.append(cand[i])
        if len(picked) == 2:
            break
    return f"'{e['topic']}' 연구에 필요한 {'·'.join(picked)}을(를) 배우는 과목이에요."

for e in evidence:
    e['M1'], e['M2'], e['M3'] = m1(e), m2(e), m3(e)
print('M1~M3 완료. 예시:')
for k in ['M1', 'M2', 'M3']:
    print(f'  {k}:', evidence[0][k])

In [ ]:
# M4: 로컬 LLM 생성
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM_ID = 'Qwen/Qwen2.5-7B-Instruct'
t0 = time.time()
tok = AutoTokenizer.from_pretrained(LLM_ID, trust_remote_code=True)
tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID, device_map='auto', trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
print(f'LLM 로드 {time.time()-t0:.1f}s')

SYSTEM = ('너는 대학생에게 연구실과 수업의 연결을 설명하는 도우미다. 아래 [근거]에 적힌 내용만 사용해 한국어 1~2문장(70자 이내 권장)으로 '
          '이 수업이 이 연구실 연구에 어떻게 도움이 되는지 설명하라. 근거에 없는 사실, 연구실 프로젝트, 합격·필수 여부를 지어내지 마라. '
          '"반드시", "필수", "최고" 같은 단정 표현을 쓰지 마라. 문장만 출력하라.')
def prompt(e):
    user = (f"[근거]\n연구실 주제: {e['topic']}\n연구실 자료 문장: {e['topic_quote']}\n"
            f"수업: {e['course']} ({e['course_en']})\n수업 자료 문장({e['where']}): {e['sentence']}")
    return tok.apply_chat_template([{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}],
                                   tokenize=False, add_generation_prompt=True)

t0 = time.time()
BATCH = 8
for i in range(0, len(evidence), BATCH):
    chunk = evidence[i:i+BATCH]
    enc_in = tok([prompt(e) for e in chunk], return_tensors='pt', padding=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**enc_in, max_new_tokens=120, do_sample=False, pad_token_id=tok.eos_token_id)
    for e, ids in zip(chunk, out[:, enc_in['input_ids'].shape[1]:]):
        e['M4'] = tok.decode(ids, skip_special_tokens=True).strip().split(chr(10))[0].strip()
    print(f'  {min(i+BATCH, len(evidence))}/{len(evidence)}')
print(f'M4 생성 {time.time()-t0:.1f}s. 예시: {evidence[0]["M4"]}')

## 3단계: 검증 (M5)과 자동 지표

- **근거에 없는 단어 비율**: 문장 속 내용어(한글 2자 이상, 영어 3자 이상) 중 근거(주제·주제 문장·과목명·과목 문장)에 없는 비율.
  한국어 조사·어미 변화 때문에 앞 2글자 일치도 허용한다. 틀 문구(연구, 과목, 배우 등)는 제외한다.
- **과장 표현**: 반드시, 필수, 최고, 완벽, 보장 등
- **M5**: M4의 근거 밖 단어 비율이 `UNSUP_MAX`보다 높거나 과장 표현이 있으면 M2로 대체

In [ ]:
FRAME = {'연구', '연구실', '과목', '수업', '배우', '배워', '배울', '다뤄', '다루', '연결', '도움', '필요', '관련', '지식', '내용',
         '이해', '학습', '활용', '기초', '바탕', '주제', '분야', '통해', '대한', '위한', '있어요', '있습니다', '있는', '하는', '돼요', '이에요', '에서'}
OVERCLAIM = re.compile(r'(반드시|필수|최고|완벽|보장|무조건|꼭 들어야)')
def content_words(s):
    return [w for w in re.findall(r'[A-Za-z]{3,}|[가-힣]{2,}', s) if w not in FRAME and not any(w.startswith(f) for f in FRAME)]
def unsupported_ratio(text, e):
    src = ' '.join([e['topic'], e['topic_quote'], e['course'], e['course_en'], e['sentence'], ' '.join(e['concepts'])]).lower()
    words = content_words(text)
    if not words:
        return 0.0
    miss = [w for w in words if w.lower() not in src and w[:2].lower() not in src]
    return len(miss) / len(words)

UNSUP_MAX = 0.35
for e in evidence:
    bad = unsupported_ratio(e['M4'], e) > UNSUP_MAX or OVERCLAIM.search(e['M4']) or len(e['M4']) < 10
    e['M5'] = m2(e) if bad else e['M4']
    e['M5_fallback'] = bool(bad)

METHODS = ['M1', 'M2', 'M3', 'M4', 'M5']
import pandas as pd
auto = {}
for m in METHODS:
    texts = [e[m] for e in evidence]
    auto[m] = {'평균 길이(자)': round(statistics.mean(len(t) for t in texts), 1),
               '근거 밖 단어 비율': round(statistics.mean(unsupported_ratio(t, e) for t, e in zip(texts, evidence)), 3),
               '과장 표현(건)': sum(bool(OVERCLAIM.search(t)) for t in texts),
               '서로 다른 문장 비율': round(len(set(texts)) / len(texts), 3)}
print(f"M5에서 M2로 대체된 비율: {sum(e['M5_fallback'] for e in evidence)/len(evidence):.0%}")
pd.DataFrame(auto)

In [ ]:
# 나란히 보기
for e in evidence[:8]:
    print(f"\n■ {e['lab']} · {e['topic']} ↔ {e['code']} {e['course']}  (근거 점수 {e['rr']})")
    print(f"   근거: {e['where']} — {short(e['sentence'], 90)}")
    for m in METHODS:
        print(f'   {m}: {e[m]}')

## 4단계: Claude Code 채점용 파일 (블라인드)

쌍마다 다섯 문장을 섞고 `r1~r5`로 이름을 바꾼다. 방법 대응표는 `reason_key.json`에만 저장한다(M5가 M2·M4와 같은 문장이면 하나로 합친다).

채점 기준:
- **faithful** 0/1 — 근거에 없는 사실·과장이 없으면 1
- **specific** 1~3 — 이 과목과 이 연구실에만 해당하는 구체적인 내용인가
- **fluent** 1~3 — 자연스러운 한국어인가
- **useful** 1~3 — 학생이 "왜 이 과목인지" 이해하는 데 도움이 되는가

In [ ]:
packet, key = [], {}
for pi, e in enumerate(evidence):
    texts = {}
    for m in METHODS:
        texts.setdefault(e[m], []).append(m)
    items = list(texts.items())
    random.Random(pi).shuffle(items)
    reasons = []
    for j, (t, ms) in enumerate(items):
        rid = f'p{pi:03d}|r{j+1}'
        reasons.append({'id': rid, 'text': t})
        key[rid] = ms
    packet.append({'pair': f'p{pi:03d}', 'lab': e['lab'], 'lab_summary': e['lab_summary'], 'topic': e['topic'],
                   'topic_quote': e['topic_quote'][:250], 'course': f"{e['code']} {e['course']} / {e['course_en']}",
                   'course_description': (course_by_id[e['record_id']].get('description') or '')[:300],
                   'evidence_sentence': e['sentence'], 'reasons': reasons})
json.dump({'instructions': '각 reason에 faithful(0/1), specific(1~3), fluent(1~3), useful(1~3). '
                           '반환 형식: {"<reason id>": {"faithful":0|1,"specific":1-3,"fluent":1-3,"useful":1-3}}',
           'pairs': packet}, open('reason_packet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(key, open('reason_key.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'채점용 파일: 쌍 {len(packet)}개, 문장 {len(key)}개 → reason_packet.json')

## 5단계 (채점 후): 결과 계산

Claude Code가 돌려준 `reason_scores.json`을 업로드하면 방법별 점수를 계산한다. 아직 없으면 업로드 창을 취소하고 넘어간다.

In [ ]:
if not os.path.exists('reason_scores.json'):
    try:
        files.upload()
    except Exception:
        pass
if os.path.exists('reason_scores.json'):
    sc = json.load(open('reason_scores.json', encoding='utf-8'))
    per = defaultdict(lambda: defaultdict(list))
    for rid, v in sc.items():
        for m in key.get(rid, []):
            for k in ['faithful', 'specific', 'fluent', 'useful']:
                per[m][k].append(v[k])
    display(pd.DataFrame({m: {k: round(statistics.mean(per[m][k]), 3) for k in per[m]} for m in METHODS if m in per}))
else:
    print('채점 결과 없음 — reason_packet.json을 Claude Code에 채점받은 뒤 이 셀을 다시 실행하세요.')

## 결과 저장·다운로드

In [ ]:
import zipfile
json.dump(evidence, open('reasons_v1.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
pd.DataFrame(auto).to_csv('reason_auto_metrics.csv', encoding='utf-8-sig')
json.dump({'N_LABS': N_LABS, 'TOP_K': TOP_K, 'SEED': SEED, 'LLM_ID': LLM_ID, 'UNSUP_MAX': UNSUP_MAX,
           'RERANKER': 'BAAI/bge-reranker-v2-m3', 'EMBEDDER': 'BAAI/bge-m3'}, open('reason_settings.json', 'w'), indent=1)
with zipfile.ZipFile('reason_results_v1.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['reasons_v1.json', 'reason_auto_metrics.csv', 'reason_settings.json', 'reason_packet.json', 'reason_key.json']:
        z.write(f)
print('저장 완료: reason_results_v1.zip')
files.download('reason_results_v1.zip')